In [ ]:
import torch

# 1. UNSQUEEZE - Adds dimension
a = torch.tensor([1, 2, 3])
print(f"Original: {a.shape}")
a = a.unsqueeze(0)
print(f"After unsqueeze: {a.shape}")

# 2. SQUEEZE - Removes dimension of size 1
b = torch.tensor([[1, 2, 3]])
print(f"Original: {b.shape}")
b = b.squeeze()
print(f"After squeeze: {b.shape}")

# 3. TOLIST - Convert to Python list
c = torch.tensor([1, 2, 3])
c_list = c.tolist()
print(f"Tensor to list: {c_list}")

# 4. DETACH - Remove gradient tracking
d = torch.tensor([1.0, 2.0], requires_grad=True)
d_detached = d.detach()
print(f"Requires grad before: {d.requires_grad}, after: {d_detached.requires_grad}")

# 5. VIEW - Reshape tensor
e = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8]])
print(f"Original shape: {e.shape}")
e = e.view(4, 2)
print(f"After view(4,2): {e.shape}")

# 6. CLONE - Create copy
f = torch.tensor([1, 2, 3])
f_clone = f.clone()
f[0] = 100
print(f"Original: {f}, Clone: {f_clone}")

# 7. RESHAPE - Reshape tensor
g = torch.tensor([[1, 2, 3], [4, 5, 6]])
g = g.reshape(3, 2)
print(f"After reshape: {g.shape}")

# 8. .TO() - Change device/dtype
h = torch.tensor([1, 2, 3], dtype=torch.float32)
h = h.to(torch.int32)
print(f"After to(int32): {h.dtype}")

Original: torch.Size([3])
After unsqueeze: torch.Size([1, 3])
Original: torch.Size([1, 3])
After squeeze: torch.Size([3])
Tensor to list: [1, 2, 3]
Requires grad before: True, after: False
Original shape: torch.Size([2, 4])
After view(4,2): torch.Size([4, 2])
Original: tensor([100,   2,   3]), Clone: tensor([1, 2, 3])
After reshape: torch.Size([3, 2])
After to(int32): torch.int32


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

# Generate binary classification data
np.random.seed(42)

# Class 0 and Class 1 data
X0 = np.random.randn(100, 2) + [2, 2]  # Class 0 around (2,2)
X1 = np.random.randn(100, 2) + [6, 6]  # Class 1 around (6,6)

X = np.vstack([X0, X1])
y = np.hstack([np.zeros(100), np.ones(100)])

# Shuffle
idx = np.random.permutation(200)
X, y = X[idx], y[idx]

# Split train/test
X_train, X_test = X[:160], X[160:]
y_train, y_test = y[:160], y[160:]

# Save to CSV
train_df = pd.DataFrame(X_train, columns=['feature1', 'feature2'])
train_df['target'] = y_train
test_df = pd.DataFrame(X_test, columns=['feature1', 'feature2'])
test_df['target'] = y_test
train_df.to_csv('train_data.csv', index=False)
test_df.to_csv('test_data.csv', index=False)

# Convert to tensors (same as linear regression)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Logistic Regression Model (similar to Linear but with Sigmoid)
model = torch.nn.Sequential(
    torch.nn.Linear(2, 1),
    torch.nn.Sigmoid()
)

# Print weights before training
print("Weights before training:")
for param in model.parameters():
    print(param.data)

# Loss and Optimizer (BCE for classification instead of MSE)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)


epochs = 10

print("\nTraining...")
print(f"{'Epoch':<6} {'Loss':<10}")

for epoch in range(epochs):
    # Forward pass
    y_pred = model(X_train_t)

    # Calculate loss
    loss = criterion(y_pred, y_train_t)

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print loss
    print(f"{epoch+1:<6} {loss.item():<10.4f}")

# Print weights after training
print("\nWeights after training:")
for param in model.parameters():
    print(param.data)


with torch.no_grad():
    test_probs = model(X_test_t)
    test_preds = (test_probs >= 0.5).float()

accuracy = accuracy_score(y_test, test_preds.numpy())
print(f"\nTest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

Weights before training:
tensor([[ 0.6496, -0.1549]])
tensor([0.1427])

Training...
Epoch  Loss      
1      0.7020    
2      0.6356    
3      0.6045    
4      0.5930    
5      0.5877    
6      0.5835    
7      0.5795    
8      0.5756    
9      0.5717    
10     0.5679    

Weights after training:
tensor([[ 0.4998, -0.2432]])
tensor([-0.0785])

Test Accuracy: 0.6000 (60.00%)
